# AeroConform - Run All Notebooks

Clones the repo, installs the package, strips Colab-specific cells,
and executes all 6 project notebooks sequentially via nbconvert.

In [ ]:
!git clone https://github.com/Nadosaurusrex/aeroconform.git /content/aeroconform 2>/dev/null || echo 'Already cloned'
%cd /content/aeroconform
!pip install -e '.[dev]' -q 2>&1 | tail -3
!pip install nbconvert ipykernel -q 2>&1 | tail -1

In [ ]:
# Verify installation (reload to pick up newly installed package)
import importlib, aeroconform
importlib.reload(aeroconform)
print(f"AeroConform v{aeroconform.__version__} installed")
from aeroconform.config import AeroConformConfig
config = AeroConformConfig()
print(f"Config: d_model={config.d_model}, n_layers={config.n_layers}, alpha={config.alpha}")

In [ ]:
# Preprocess notebooks: strip !pip install and google.colab cells
import json, glob, os

os.makedirs('/content/aeroconform/notebooks/_run', exist_ok=True)

for nb_path in sorted(glob.glob('/content/aeroconform/notebooks/0*.ipynb')):
    with open(nb_path) as f:
        nb = json.load(f)
    clean = []
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            src = ''.join(cell['source'])
            if '!pip' in src or 'google.colab' in src or 'drive.mount' in src:
                continue
        clean.append(cell)
    nb['cells'] = clean
    out = f"/content/aeroconform/notebooks/_run/{os.path.basename(nb_path)}"
    with open(out, 'w') as f:
        json.dump(nb, f)
    print(f'{os.path.basename(nb_path)}: {len(clean)} cells')
print('All notebooks preprocessed')

In [ ]:
# Quick test run to verify package works
!cd /content/aeroconform && python -m pytest tests/test_geo.py tests/test_tokenizer.py -v --tb=short 2>&1 | tail -15

## Execute All 6 Notebooks

In [ ]:
import subprocess, time

notebooks = [
    '01_data_exploration.ipynb',
    '02_pretraining.ipynb',
    '03_graph_training.ipynb',
    '04_conformal_calibration.ipynb',
    '05_evaluation.ipynb',
    '06_live_demo.ipynb',
]

results = {}
for nb in notebooks:
    src = f'/content/aeroconform/notebooks/_run/{nb}'
    out = f'/content/executed_{nb}'
    print(f'\n{"="*60}')
    print(f'Running: {nb}')
    start = time.time()
    r = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'notebook', '--execute',
         '--ExecutePreprocessor.timeout=300',
         '--ExecutePreprocessor.kernel_name=python3',
         src, '--output', out],
        capture_output=True, text=True, cwd='/content/aeroconform'
    )
    elapsed = time.time() - start
    ok = r.returncode == 0
    results[nb] = {'ok': ok, 'time': elapsed}
    status = 'PASS' if ok else 'FAIL'
    print(f'{status} ({elapsed:.1f}s)')
    if not ok:
        print(r.stderr[-500:] if r.stderr else 'no stderr')

print(f'\n{"="*60}')
print('SUMMARY')
for nb, r in results.items():
    print(f"  {'PASS' if r['ok'] else 'FAIL'} {nb} ({r['time']:.1f}s)")
passed = sum(1 for r in results.values() if r['ok'])
print(f'\n{passed}/{len(results)} notebooks passed')